# LinAge2

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.LinAge2)

class LinAge2(pyagingModel):
    """Principal-component clinical clock trained on NHANES IV mortality (Fong et al. 2025)."""

    def __init__(self):
        super().__init__()
        # The 59 names the loadings, medians and MADs are indexed by, in SVD row order.
        # Set from clocks/linage2_params.json when the clock is built.
        self.model_features = None
        for sex in ["male", "female"]:
            for name in ["median", "mad", "loadings", "beta", "means", "beta_null", "mean_null", "mrdt"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))
            self.register_buffer(f"pc_index_{sex}", torch.empty(0, dtype=torch.long))
        self.register_buffer("log_mask", torch.empty(0, dtype=torch.bool))
        self.register_buffer("skip_mask", torch.empty(0, dtype=torch.bool))

    def preprocess(self, x):
        return x

    def _model_vector(self, x):
        """Assemble the 59-feature vector from the user-facing inputs.

        Notes
        ---

In [3]:
model = pya.models.LinAge2()

## Define clock metadata

Each `# Paper:` comment reproduces the evidence recorded for that field in `clocks/metadata/evidence_ledger.jsonl`; `validate_metadata.py` compares the two, so they cannot drift apart. The paper is open access (CC-BY). Two fields are sourced from the `linAge2.R` script shipped as supplementary material rather than from the article text: `tissue`, because only the script shows that two of the model features are urine assays, and `n_features`, because the article counts the internal model vector while pyaging declares the user-facing inputs.

`tissue` uses the vocabulary value `urine`, which this commit adds to `clocks/metadata/controlled_vocabulary.json` — LinAge2 is the first packaged clock built on a urine assay.

In [4]:
model.metadata["clock_name"] = "linage2"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: we refined the clinical parameters by reducing the total number to 60
model.metadata["species"] = "Homo sapiens"  # Paper: Not all humans age at the same rate since genetics, lifestyle, and stochastic factors significantly affect future mortality and morbidity trajectories.
model.metadata["year"] = 2025
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Fong, Sheng, et al. \"LinAge2: providing actionable insights and benchmarking with epigenetic clocks.\" npj Aging 11.1 (2025): 29."
model.metadata["doi"] = "https://doi.org/10.1038/s41514-025-00221-4"
model.metadata["notes"] = "Principal-component clinical clock trained on 20-year mortality in the NHANES IV 1999-2000 wave and tested in the 2001-2002 wave. The 59 model features are log transformed where the reference specifies, robustly z-scored by sex against a healthy 40-50 year old NHANES reference, and folded at 6 MAD-scaled units; the folded vector is projected onto sex-specific singular vectors and scored by a sex-specific Cox model, so male and female samples run through entirely separate parameter sets. Chronological age is supplied in years and enters the Cox terms in months, where it is a genuine covariate rather than a cancelling offset. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters. C-reactive protein is supplied raw in mg/dL and takes a plain natural log with no floor, so a reading below detection coded as 0 folds to the -6 cap instead of being clamped the way the BioAge clocks clamp it. The input contract is wider than the 59 model features: total cholesterol, HDL cholesterol and triglycerides are consumed only by the Friedewald LDL and are not features themselves, and 26 questionnaire codes feed the comorbidity, self-reported-health and healthcare-use indices. An absent questionnaire block is the main hazard and it biases the estimate downward. The substituted values are the reference cohort's median profile everywhere except the comorbidity index: answering no to all 22 conditions gives 0 where the cohort's median is 1/22, so that one feature is substituted marginally healthier than the median. It costs almost nothing, because the index's entire 0 to 1 range moves the estimate by only 0.06 years. The self-reported-health index is what does the damage: its substitute of good, unchanged health is exactly the cohort median, and a subject who would have reported poor and worsening health reads about 5.3 years younger than they should. The healthcare-use index pushes the other way, by about 0.5 years for a subject who made 16 or more visits. An absent lipid panel substitutes a total cholesterol chosen so that the derived LDL lands on the reference median, avoiding the reference implementation's hard 0 mmol/L substitution; a NaN inside a lipid column that is present still takes that hard 0 path and lowers the estimate by roughly 0.35 years. Heed the missing-feature warning the prediction pipeline emits."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood", "urine"]  # Paper: albuVals <- dataMat[,"URXUMASI"] crAlbRat <- albuVals/(creaVals*1.1312*10^-4)
model.metadata["predicts"] = ["biological age"]  # Paper: computational tools that estimate individual true BA based on demographic, clinical, and/or molecular data
model.metadata["training_target"] = ["mortality"]  # Paper: clocks trained on survival and functional aging outperform those trained on chronological age
model.metadata["unit"] = ["years"]  # Paper: had BA deltas of at most 35 years with estimated BAs that never exceeded 105 years
model.metadata["model_type"] = "PCA + Cox regression"  # Paper: The loadings for male and female PCs are provided in Supplementary Table, and sex-specific weights of the Cox proportional hazards models are listed in Supplementary Table.
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: further refining the clinical parameters, especially removing serum fibrinogen due to the need for a specialized sodium citrate tube
model.metadata["population"] = "adults"  # Paper: we excluded participants top-coded at age 85 years, as we could not ascertain the exact CAs of these adults
model.metadata["journal"] = "npj Aging"
model.metadata["last_author"] = "Jan Gruber"
model.metadata["n_features"] = 85
model.metadata["citations"] = 6
model.metadata["citations_date"] = "2026-08-20"

## Download clock dependencies

The reference implementation re-derives every constant at run time by re-fitting the training pipeline over the NHANES 1999-2002 merged file, so nothing is shipped as a saved parameter. The constants were therefore extracted once from an instrumented run of `linAge2.R` and are checked in under `clocks/linage2_params.json`: the 59-element feature order with its log and skip masks, and, per sex, the reference median and MAD, the 59x59 matrix of right singular vectors, the Cox coefficients and training means with the PC numbers they belong to, and the null model's coefficient, mean and pre-rounded mortality rate doubling time.

In [5]:
with open("../linage2_params.json") as handle:
    params = json.load(handle)

params["features"]

['pulse',
 'systolic_blood_pressure',
 'diastolic_blood_pressure',
 'body_mass_index',
 'urine_albumin',
 'urine_creatinine',
 'iron',
 'total_iron_binding_capacity',
 'transferrin_saturation',
 'ferritin',
 'folate',
 'vitamin_b12',
 'smoking_intensity',
 'white_blood_cell_count',
 'lymphocyte_percent',
 'monocyte_percent',
 'neutrophil_percent',
 'eosinophil_percent',
 'basophil_percent',
 'lymphocyte_count',
 'monocyte_count',
 'neutrophil_count',
 'eosinophil_count',
 'basophil_count',
 'red_blood_cell_count',
 'hemoglobin',
 'hematocrit',
 'mean_cell_volume',
 'mean_cell_hemoglobin',
 'mean_cell_hemoglobin_concentration',
 'red_cell_distribution_width',
 'platelet_count',
 'mean_platelet_volume',
 'c_reactive_protein',
 'hemoglobin_a1c',
 'nt_probnp',
 'albumin',
 'alanine_aminotransferase',
 'aspartate_aminotransferase',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'calcium',
 'bicarbonate',
 'glucose',
 'lactate_dehydrogenase',
 'phosphorus',
 'total_bilirubin',
 'total_pro

## Load features

`model.features` is the **user-facing input contract**, not the 59-element model vector. It is wider in both directions: total cholesterol, HDL cholesterol and triglycerides are consumed only by the Friedewald LDL and are not model features, cotinine is digitized into a smoking-intensity code before it becomes one, and the 26 questionnaire items collapse into just three of them. `postprocess` derives the six computed quantities and assembles the 59-vector itself.

In [6]:
model.features = params["inputs"]["numeric"] + params["inputs"]["questionnaire"] + ["age", "female"]
len(model.features), model.features[:5], model.features[-5:]

(85,
 ['pulse',
  'systolic_blood_pressure',
  'diastolic_blood_pressure',
  'body_mass_index',
  'urine_albumin'],
 ['fractured_spine',
  'told_osteoporosis',
  'confusion_or_memory_problems',
  'age',
  'female'])

#### Normal feature ranges

Every feature above is registered in `pyaging`'s feature range registry, which is the single source of truth for units and plausible bounds; the clock stores those units in `model.feature_units`, so a saved clock is self-describing and carries its own units even if the package registry later standardizes differently. `tests/test_clock_metadata.py` asserts every built clock's stored copy still matches the registry, so a registry correction cannot be silently shadowed by a stale one. `pya.utils.get_feature_ranges("linage2")` reports them for a saved clock.

Units worth calling out, because the reference's NHANES codebook and pyaging's SI convention agree here only by luck:

- `c_reactive_protein` is in **mg/dL**, as NHANES 1999-2000 reported it, and is natural-log transformed inside the clock with **no floor**. A reading of `0` therefore reaches the fold as `-inf` and becomes exactly `-6`, which is what the reference does; the shared `log1p_crp` helper the BioAge clocks use would clamp it to 0.01 mg/dL instead and give a different answer, so this clock does not use it.
- `albumin` is **g/L**, `blood_urea_nitrogen`, `glucose` and `calcium` are **mmol/L**, `creatinine` and `urine_creatinine` are **µmol/L**, and `hemoglobin` is **g/dL**.
- `cotinine` is raw serum cotinine in **ng/mL**, not a smoking-status code. The clock bins it at 10, 100 and 200 ng/mL into the 0-3 intensity the model actually sees, so handing it a status code would be read as a cotinine concentration.
- `age` is in **years**. The Cox terms need months, and `postprocess` multiplies by 12.
- `female` is `1` for female and `0` for male. A dataset with no `female` column scores every sample with the male parameters — different loadings, different Cox weights and a different mortality rate doubling time, not a small correction.
- The questionnaire columns are raw NHANES codes, where `1` means yes and `2` means no for the yes/no items. `told_diabetes` also counts `3` (borderline) as a yes.
- `self_reported_health_index` and `healthcare_use_index` are registered over their 0-8 input range, but they skip z-scoring and still meet the ±6 fold, so 7 and 8 both enter the projection as 6.

In [ ]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

## Load weights into base model

There is nothing to learn here either: LinAge2 is a fixed chain of transforms over the extracted constants, so the base model is the identity and all of the arithmetic lives in `LinAge2.postprocess`. The constants are stored as buffers, one set per sex, plus the two shared masks.

`model_features` is the 59-name order the loadings, medians and MADs are indexed by. It is load-bearing: the projection is a plain right-multiplication by a 59x59 matrix whose rows are those features in that order, and a permutation would be silent.

In [8]:
model.base_model = torch.nn.Identity()

model.model_features = params["features"]
model.log_mask = torch.tensor(params["log_mask"], dtype=torch.bool)
model.skip_mask = torch.tensor(params["skip_mask"], dtype=torch.bool)

for sex in ["male", "female"]:
    fit = params[sex]
    for key in ["median", "mad", "loadings", "beta", "means", "beta_null", "mean_null", "mrdt"]:
        setattr(model, f"{key}_{sex}", torch.tensor(fit[key], dtype=torch.float64))
    setattr(model, f"pc_index_{sex}", torch.tensor(fit["pc_index"], dtype=torch.long))
    # mrdt is round(ln(2) / beta_null, 2) in the reference. The rounding is load-bearing, so the
    # stored value is used as-is; recomputing it unrounded moves the fourth decimal of the age.
    assert fit["mrdt"] == round(math.log(2) / fit["beta_null"], 2)

# Every name the 59-vector needs is either a user-facing input or one of the six quantities
# postprocess derives, and the comorbidity item list must be the same 22 the JSON records.
assert set(model.model_features) - set(model.features) == set(params["derived"])
assert set(pya.models._models.LINAGE2_COMORBIDITY_ITEMS) == set(
    params["derived"]["comorbidity_index"]["inputs"]
)
assert len(pya.models._models.LINAGE2_COMORBIDITY_ITEMS) == 22

model.mrdt_male, model.mrdt_female

(tensor(103.9500, dtype=torch.float64), tensor(82.4300, dtype=torch.float64))

## Load reference values

`check_features_in_adata` substitutes these for any feature a user's dataframe does not carry. Zero-filling would be badly wrong here — a zero assay z-scores to tens of MADs and then pins at the fold, dragging every principal component with it — so each input is instead set to the value that puts its model feature on the **young NHANES reference median**, the same reference the z-scores are taken against.

The medians are sex-specific and `reference_values` is one vector, so the mean of the male and female medians is used. For the 13 log-transformed features the median lives on the log scale, so `exp` is applied to bring it back to the units the user supplies.

The derived features need their *inputs* chosen so that the derived value lands on the median:

- **cotinine** → `0.0`. The reference cohort's median smoking intensity is `0`, and any value under 10 ng/mL digitizes to it.
- **the questionnaire block** → the NHANES codes for the median profile: `2` (no) for all 22 comorbidity items, `3`/`3` (good, unchanged) for the two self-reported-health items, and the median visit code for healthcare use. The first two are the reference implementation's own NA defaults; only one of them is exactly the reference median. The healthy 40-50 year old cohort's median self-reported-health index is `0`, which is what `3`/`3` gives. Its median comorbidity index is `1/22`, and answering `2` to all 22 items gives `0` — so that one substitute is marginally *healthier* than the median. Landing on `1/22` would mean marking one arbitrary condition as `yes`, which claims a disease the subject never reported, and the index's entire 0-to-1 range is worth only 0.059 years, so the honest substitute wins. Healthcare use is the other place the reference's default and the median part ways — it defaults to 0 visits where the median is 2 — and there the median is used, because substituting 0 instead moves the paper's first example subject by 0.23 years.
- **the lipid panel** → total cholesterol, HDL and triglycerides only ever reach the model through `total_cholesterol - triglycerides / 5 - hdl_cholesterol`, so HDL and triglycerides are set to mid-normal adult concentrations (each substituted value has to be plausible on its own, in case only one of the three is absent) and total cholesterol is then solved to put the derived LDL on the reference median. This is the one place the port deliberately improves on the reference, which substitutes a hard `0 mmol/L` LDL — a physiologically absurd value that z-scores to about -3.6 for a male and costs 0.35 years on the paper's first example subject.

Two inputs have no neutral value. `age` is set to `62`, the midpoint of the reference script's 40-84 training window, because the age term is a genuine Cox covariate. `female` is set to `0`, which means a dataset with no sex column is scored entirely with the male parameter set. Both are recorded in the metadata notes.

**What this does not fix.** A subject who *would* have reported poor and worsening health gets the median profile instead, and reads about 5.3 years younger for it. The bias is downward and silent, and no choice of a single substitute value can avoid it — the median of the healthy reference cohort simply is the healthy answer. `tests/predict/test_linage2.py` pins the magnitude and direction, and the metadata notes state them.

In [9]:
LIPID_HDL = 1.3  # mmol/L, mid-normal adult
LIPID_TRIGLYCERIDES = 1.3  # mmol/L, mid-normal adult

reference_median = {
    name: math.exp((male + female) / 2) if logged else (male + female) / 2
    for name, male, female, logged in zip(
        params["features"], params["male"]["median"], params["female"]["median"], params["log_mask"]
    )
}

substitute = {name: 2.0 for name in params["inputs"]["questionnaire"]}  # 2 = no
substitute.update(
    {
        "general_health_condition": 3.0,  # good
        "health_compared_to_one_year_ago": 3.0,  # about the same
        "healthcare_visits_past_year": reference_median["healthcare_use_index"],
        "cotinine": 0.0,  # any value under 10 ng/mL digitizes to intensity 0
        "hdl_cholesterol": LIPID_HDL,
        "triglycerides": LIPID_TRIGLYCERIDES,
        "total_cholesterol": reference_median["ldl_cholesterol"] + LIPID_TRIGLYCERIDES / 5 + LIPID_HDL,
        "age": 62.0,  # midpoint of the 40-84 training window
        "female": 0.0,  # no sex column means the male parameters
    }
)

model.reference_values = [substitute.get(name, reference_median.get(name)) for name in model.features]

assert len(model.reference_values) == len(model.features)
assert None not in model.reference_values

# Every substituted value has to be plausible on its own, not merely neutral in aggregate.
for record, value in zip(
    pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"]), model.reference_values
):
    assert record["low"] <= value <= record["high"], (record["feature"], value)

pd.DataFrame({"feature": model.features, "reference": model.reference_values})

,feature,reference
0,pulse,71.000000
1,systolic_blood_pressure,120.000000
2,diastolic_blood_pressure,76.500000
3,body_mass_index,28.077449
4,urine_albumin,7.300000
...,...,...
80,fractured_spine,2.000000
81,told_osteoporosis,2.000000
82,confusion_or_memory_problems,2.000000
83,age,62.000000


## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "linage2"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Fong, Sheng, et al. "LinAge2: providing actionable insights and '
             'benchmarking with epigenetic clocks." npj Aging 11.1 (2025): 29.',
 'citations': 6,
 'citations_date': '2026-08-20',
 'clock_name': 'linage2',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1038/s41514-025-00221-4',
 'journal': 'npj Aging',
 'last_author': 'Jan Gruber',
 'model_type': 'PCA + Cox regression',
 'n_features': 85,
 'notes': 'Principal-component clinical clock trained on 20-year mortality in '
          'the NHANES IV 1999-2000 wave and tested in the 2001-2002 wave. The '
          '59 model features are log transformed where the reference '
          'specifies, robustly z-scored by sex against a healthy 40-50 year '
          'old NHANES reference, and folded at 6 MAD-scaled units; the folded '
       

## Basic test

The smoke test feeds the midpoint of each feature's registered range. Those bounds are deliberately wide — they are the range a value may plausibly take, not the range it usually takes — so the midpoint subject is an implausibly unwell one and lands near the top of the scale. It checks that the chain runs and stays finite, nothing more.

In [13]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[109.6068]], dtype=torch.float64)

#### Parity with the published example subjects

The acceptance gate is `tests/predict/test_linage2.py`, which reproduces the two biological ages the paper prints for its example NHANES subjects (SEQN 8881 → 88.69 y, SEQN 9106 → 64.36 y). Reproduced inline here as well, since a notebook that assembles constants should show that they land where the source lands. The reference rounds its output to two decimals; pyaging does not, so the residual below is that rounding.

In [14]:
rows = [dict(case["inputs"], age=case["age_years"], female=float(case["female"])) for case in params["validation"]]

matrix = torch.tensor(
    [[float(row[name]) for name in model.features] for row in rows], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor([case["biological_age"] for case in params["validation"]], dtype=torch.float64)
print("predicted:", predicted.tolist())
print("published:", expected.tolist())
print("max absolute difference:", (predicted - expected).abs().max().item())

predicted: [88.69448735215042, 64.35789927322698]
published: [88.69, 64.36]
max absolute difference: 0.004487352150420065


## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)